# Bucks intent model: convert run 2 to `.litertlm` (CPU)
Input: the Output of notebook `bucks-intent-run2` (Add Input → Your Work). Internet on, no GPU needed. Save & Run All; the Output tab then has `bucks-intent_q8_ekv1024.litertlm` and `eval_run2.json`.

In [ ]:
# 1. Find run 2's checkpoint zip and unpack it
import glob, zipfile, os, shutil
z = glob.glob('/kaggle/input/**/bucks-intent-run2.zip', recursive=True)
print(z)
assert z, 'Add Input: the Output of notebook bucks-intent-run2'
zipfile.ZipFile(z[0]).extractall('ckpt')
shutil.copy('ckpt/eval_run2.json', 'eval_run2.json')
print(sorted(os.listdir('ckpt')))

In [ ]:
# 2. The converter script (packed), fixed quantize recipe
import base64, hashlib
raw = base64.b64decode('IiIiQ29udmVydCB0aGUgZmluZS10dW5lZCBjaGVja3BvaW50IHRvIGEgLmxpdGVydGxtIGZvciBMaXRlUlQtTE0gb24gdGhlIHBob25lLiBDUFUgb25seSwgbm8gR1BVIG5lZWRlZC4KClJ1bnMgaW4gdGhyZWUgc2VwYXJhdGUgcHJvY2Vzc2VzIChleHBvcnQsIHF1YW50aXplLCBwYWNrKSBzbyB0aGUgcGVhayBtZW1vcnkgb2YgZWFjaCBzdGVwIHN0YXlzIHVuZGVyIH40IEdCLgpOZWVkcyBhIHZlbnYgd2l0aDogdG9yY2g8Mi4xNSAoQ1BVIGJ1aWxkKSwgbGl0ZXJ0LXRvcmNoPT0wLjEwLjAsIHRlbnNvcmZsb3ctY3B1IChpbXBvcnRlZCBidXQgbm90IGRlY2xhcmVkKSwgamF4PT0wLjEyLjAuMQphbmQgamF4bGliPT0wLjEyLjAuMSAoMC4xMi4xIGJyZWFrcyB0aGUgZXhwb3J0KSwgbGl0ZXJ0LWxtLWJ1aWxkZXI9PTAuMTguMSAoMC4xOSBkcm9wcGVkIFRmTGl0ZU1vZGVsVHlwZSkuClVzYWdlOiBweXRob24gLUkgY29udmVydF9saXRlcnRsbS5weSB7ZXhwb3J0fHF1YW50aXplfHBhY2t9IENIRUNLUE9JTlRfRElSIE9VVF9ESVIKT3V0cHV0OiBPVVRfRElSL2J1Y2tzLWludGVudF9xOF9la3YxMDI0LmxpdGVydGxtIChkeW5hbWljIGludDgsIDEwMjQtdG9rZW4gY29udGV4dDsgdGhlIDEyLXRvb2wgcHJvbXB0IGlzIH42MTMgdG9rZW5zKS4KIiIiCmltcG9ydCBnbG9iCmltcG9ydCBzeXMKCnN0ZXAsIGNrLCBvdXQgPSBzeXMuYXJndlsxXSwgc3lzLmFyZ3ZbMl0sIHN5cy5hcmd2WzNdClE4ID0gZiJ7b3V0fS9idWNrcy1pbnRlbnRfcThfZWt2MTAyNC50ZmxpdGUiCmlmIHN0ZXAgPT0gImV4cG9ydCI6CiAgICBmcm9tIGxpdGVydF90b3JjaC5nZW5lcmF0aXZlLmV4YW1wbGVzLmdlbW1hMyBpbXBvcnQgZ2VtbWEzCiAgICBmcm9tIGxpdGVydF90b3JjaC5nZW5lcmF0aXZlLmxheWVycyBpbXBvcnQga3ZfY2FjaGUKICAgIGZyb20gbGl0ZXJ0X3RvcmNoLmdlbmVyYXRpdmUudXRpbGl0aWVzIGltcG9ydCBjb252ZXJ0ZXIKICAgIGZyb20gbGl0ZXJ0X3RvcmNoLmdlbmVyYXRpdmUudXRpbGl0aWVzLmV4cG9ydF9jb25maWcgaW1wb3J0IEV4cG9ydENvbmZpZwogICAgY2ZnID0gRXhwb3J0Q29uZmlnKCk7IGNmZy5rdmNhY2hlX2xheW91dCA9IGt2X2NhY2hlLktWX0xBWU9VVF9UUkFOU1BPU0VEOyBjZmcubWFza19hc19pbnB1dCA9IFRydWUKICAgIHByaW50KCJleHBvcnRlZCIsIGNvbnZlcnRlci5jb252ZXJ0X3RvX3RmbGl0ZShnZW1tYTMuYnVpbGRfbW9kZWxfMjcwbShjayksIG91dCwgImJ1Y2tzLWludGVudCIsIDI1NiwgMTAyNCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBxdWFudGl6ZT0ibm9uZSIsIGV4cG9ydF9jb25maWc9Y2ZnKSkKZWxpZiBzdGVwID09ICJxdWFudGl6ZSI6CiAgICBmcm9tIGxpdGVydF90b3JjaC5nZW5lcmF0aXZlLmV4YW1wbGVzLmdlbW1hMyBpbXBvcnQgZGVjb2RlcgogICAgZnJvbSBsaXRlcnRfdG9yY2guZ2VuZXJhdGl2ZS5xdWFudGl6ZSBpbXBvcnQgcXVhbnRfcmVjaXBlcwogICAgZnJvbSBsaXRlcnRfdG9yY2gucXVhbnRpemUgaW1wb3J0IHRyYW5zbGF0ZV9yZWNpcGUKICAgIHNyYyA9IGdsb2IuZ2xvYihmIntvdXR9L2J1Y2tzLWludGVudF8qZjMyKi50ZmxpdGUiKVswXQogICAgcmVjaXBlID0gdHJhbnNsYXRlX3JlY2lwZS50cmFuc2xhdGVfdG9fYWlfZWRnZV9yZWNpcGUocXVhbnRfcmVjaXBlcy5mdWxsX2R5bmFtaWNfcmVjaXBlKG1jZmc9ZGVjb2Rlci5nZXRfZGVjb2Rlcl9jb25maWdfMjcwbSgpKS5nZW5lcmF0aXZlX3JlY2lwZSkKICAgIHEgPSB0cmFuc2xhdGVfcmVjaXBlLnF1YW50aXplX21vZGVsKG9wZW4oc3JjLCAicmIiKS5yZWFkKCksIHJlY2lwZSkKICAgIG9wZW4oUTgsICJ3YiIpLndyaXRlKHEpOyBwcmludCgicXVhbnRpemVkIiwgbGVuKHEpKQplbGlmIHN0ZXAgPT0gInBhY2siOgogICAgaW1wb3J0IHRlbXBmaWxlCiAgICBmcm9tIGxpdGVydF90b3JjaC5nZW5lcmF0aXZlLnV0aWxpdGllcyBpbXBvcnQgbGl0ZXJ0bG1fYnVpbGRlcgogICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB3ZDoKICAgICAgICBsaXRlcnRsbV9idWlsZGVyLmJ1aWxkX2xpdGVydGxtKAogICAgICAgICAgICB0ZmxpdGVfbW9kZWxfcGF0aD1ROCwgd29ya2Rpcj13ZCwgb3V0cHV0X3BhdGg9b3V0LCBjb250ZXh0X2xlbmd0aD0xMDI0LAogICAgICAgICAgICBoZl90b2tlbml6ZXJfbW9kZWxfcGF0aD1mIntja30vdG9rZW5pemVyLmpzb24iLCBsbG1fbW9kZWxfdHlwZT0iZ2VuZXJpYyIsCiAgICAgICAgICAgIHN0YXJ0X3Rva2VuPSI8Ym9zPiIsIHN0b3BfdG9rZW5zPVsiPGVuZF9vZl90dXJuPiIsICI8c3RhcnRfZnVuY3Rpb25fcmVzcG9uc2U+IiwgIjxlb3M+Il0sCiAgICAgICAgICAgIHVzZXJfcHJvbXB0X3ByZWZpeD0iPHN0YXJ0X29mX3R1cm4+dXNlclxuIiwgdXNlcl9wcm9tcHRfc3VmZml4PSI8ZW5kX29mX3R1cm4+XG4iLAogICAgICAgICAgICBtb2RlbF9wcm9tcHRfcHJlZml4PSI8c3RhcnRfb2ZfdHVybj5tb2RlbFxuIiwgbW9kZWxfcHJvbXB0X3N1ZmZpeD0iPGVuZF9vZl90dXJuPlxuIiwKICAgICAgICAgICAgamluamFfcHJvbXB0X3RlbXBsYXRlPW9wZW4oZiJ7Y2t9L2NoYXRfdGVtcGxhdGUuamluamEiKS5yZWFkKCkpCiAgICBwcmludCgicGFja2VkIikK')
assert hashlib.sha256(raw).hexdigest().startswith('c6eaaaabe585ce07')
open('convert_litertlm.py', 'wb').write(raw)

In [ ]:
# 3. Converter environment, pinned (jax 0.12.1 and litert-lm-builder 0.19 break it)
!pip install -q uv
!uv venv -q /kaggle/working/conv_env
!uv pip install -q --python /kaggle/working/conv_env/bin/python "torch<2.15" --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /kaggle/working/conv_env/bin/python "litert-torch==0.10.0" tensorflow-cpu "jax==0.12.0.1" "jaxlib==0.12.0.1" "litert-lm-builder==0.18.1"

In [ ]:
# 4. Export, quantize to int8, pack
!mkdir -p litertlm
!for s in export quantize pack; do /kaggle/working/conv_env/bin/python convert_litertlm.py $s ckpt litertlm 2>&1 | grep -vE "_pytree|KernelPreference|Failed to load" | tail -5; done
!ls -la litertlm

In [ ]:
# 5. Keep only the phone model and the test results in the Output
!rm -rf conv_env ckpt litertlm/*.tflite
!ls -la . litertlm